# Reproducing Reviewer #2's bugs in `kfre` 0.1.17

This notebook reproduces, on the **published 0.1.17 code**, the four defects Reviewer #2 reported for SOFTX-D-26-00673. Run the cells top to bottom.

**Setup:** install the exact published version so you are testing what the reviewer tested.

In [ ]:
# Install the exact published version the reviewer used

import kfre
import kfre.main as m
import numpy as np, pandas as pd
print("kfre version:", kfre.__version__)

## Bug 1 - `UnboundLocalError`

Supply the diabetes/hypertension pair **and** all four biochemical values at once. The branch selector picks the 6-variable model, but the later 8-variable risk-score block still runs and references coefficients that were never defined.

In [ ]:
try:
    m.risk_pred(
        age=60, sex=1, eGFR=25, uACR=200, is_north_american=False,
        dm=1, htn=1,
        albumin=3.5, phosphorous=4.0, bicarbonate=24, calcium=9.0,
        years=2,
    )
    print("Returned normally - bug NOT present")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# Expected on 0.1.17:
# UnboundLocalError: cannot access local variable 'albumin_factor' ...

## Bug 2 - silent fall-through for unsupported `num_vars`

`predict_kfre`'s docstring says it raises `ValueError` for an unsupported
`num_vars`. With `use_extra_vars=True, num_vars=5` it matches neither the
`==6` nor the `==8` branch, and there is no `else`, so it returns `None`.

In [ ]:
cols = {'age':'Age','sex':'Sex','eGFR':'eGFR','uACR':'Albumin_Ratio',
        'region':'Region','dm':'Diabetes','htn':'Hypertension'}
df = pd.DataFrame({"Age":[60],"Sex":["male"],"eGFR":[25],"Albumin_Ratio":[200],
                   "Region":[0],"Diabetes":[1],"Hypertension":[1]})

out = m.RiskPredictor(df=df, columns=cols).predict_kfre(
    years=2, is_north_american=False, use_extra_vars=True, num_vars=5)

print("returned:", repr(out))
# Expected on 0.1.17:  returned: None   (should have raised ValueError)

## Bug 3 - sex strings silently coerced

Sex is parsed as an exact equality (`== female_str`, or `.str.lower()=="male"`). Anything that is not an exact match, typos, `"unknown"`, empty strings, is silently bucketed into the other category instead of being rejected.

In [ ]:
vals = ["Female", "M", "man", "unknown", "", "Femal", "MALE"]
flag = (pd.Series(vals) == "Female").astype(int)

for v, f in zip(vals, flag):
    print(f"{v!r:>10}  -> treated_as_female = {f}")

# Expected on 0.1.17: only exact 'Female' -> 1; everything else silently 0,
# including the typo 'Femal', '', and 'unknown'. Nothing is rejected.

## Bug 4 - negative uACR silently clamped

`risk_pred` runs `uACR = np.maximum(uACR, 1e-6)`, so a negative (clinically impossible) uACR is silently replaced with `1e-6` and produces a normal-looking probability with no warning or error.

In [ ]:
neg  = m.risk_pred(age=60, sex=0, eGFR=25, uACR=-500,  is_north_american=False, years=2)
tiny = m.risk_pred(age=60, sex=0, eGFR=25, uACR=1e-6,  is_north_american=False, years=2)

print(f"uACR = -500  -> {neg:.6f}")
print(f"uACR = 1e-6  -> {tiny:.6f}")
print("identical:", np.isclose(neg, tiny))

# Expected on 0.1.17: identical outputs -> the negative value was silently
# accepted and clamped, turning invalid data into a valid-looking prediction.

## Summary

| Bug | Reported behavior | Confirmed on 0.1.17 |
|-----|-------------------|---------------------|
| 1 | `UnboundLocalError` when all variables supplied | yes |
| 2 | `num_vars=5` returns `None` instead of `ValueError` | yes |
| 3 | Invalid sex strings silently coerced, not rejected | yes |
| 4 | Negative uACR silently clamped to `1e-6` | yes |

All four reproduce on the published version. Fixes are the next step.

---
# Part 2 - Reproducing through the **public API**

Part 1 exercised the internal `risk_pred`. Reviewers and users, however, call the **exported** functions (`kfre_person`, `predict_kfre`, `add_kfre_risk_col`, `upcr_uacr`). This section shows which defects are reachable from that public surface, since that is what matters for the response letter.

Key finding: **Bugs 1, 2, and 4 fail through the public functions.** Bug 3 is narrower than reported, the single-patient `kfre_person` already rejects a bad sex value; only the batch `upcr_uacr` path silently coerces.

In [ ]:
from kfre import kfre_person, predict_kfre, add_kfre_risk_col, upcr_uacr
import numpy as np, pandas as pd

### Bug 1 through `kfre_person` (public, single-patient)
Pass diabetes/hypertension **and** the four biochemical values together.

In [ ]:
try:
    r = kfre_person(age=60, is_male=True, eGFR=25, uACR=200,
                    is_north_american=False, years=2,
                    dm=1, htn=1, albumin=3.5, phosphorous=4.0,
                    bicarbonate=24, calcium=9.0)
    print("returned:", r)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
# Expected: UnboundLocalError: ... 'albumin_factor' ...  (user-facing crash)

### Bug 2 through `predict_kfre` (public wrapper)

In [ ]:
cols = {'age':'Age','sex':'Sex','eGFR':'eGFR','uACR':'uACR',
        'region':'Region','dm':'DM','htn':'HTN'}
df2 = pd.DataFrame({"Age":[60],"Sex":["male"],"eGFR":[25],"uACR":[200],
                    "Region":[0],"DM":[1],"HTN":[1]})
out = predict_kfre(df2, cols, years=2, is_north_american=False,
                   use_extra_vars=True, num_vars=5)
print("returned:", repr(out))
# Expected: None  (should raise ValueError per the docstring)

### Bug 3 - the important distinction

**Single-patient `kfre_person` does NOT silently coerce.** A bad sex value raises downstream, so that public path is already safe. The silent coercion exists only in the batch `upcr_uacr` function.

In [ ]:
# 3a: kfre_person rejects a bad sex value (raises) - NOT silently coerced
try:
    kfre_person(age=60, is_male="Femal", eGFR=25, uACR=200,
                is_north_american=False, years=2)
    print("returned (silent coercion)")
except Exception as e:
    print(f"kfre_person('Femal'): {type(e).__name__}: {e}")

print()
# 3b: batch upcr_uacr DOES silently bucket blank / typo / NaN as not-female
dfx = pd.DataFrame({"SEX":["Female","Femal","",np.nan,"male"],
                    "DM":[0]*5, "HTN":[0]*5, "UPCR":[500]*5})
res = upcr_uacr(dfx, sex_col="SEX", diabetes_col="DM", hypertension_col="HTN",
                upcr_col="UPCR", female_str="Female")
for s, v in zip(dfx["SEX"], res):
    print(f"  SEX={str(s)!r:>8} -> uACR={v:.2f}")
# Expected: only 'Female' differs; '', NaN, 'Femal', 'male' all identical, no warning

### Bug 4 through `kfre_person` (public, single-patient)

In [ ]:
neg  = kfre_person(age=60, is_male=False, eGFR=25, uACR=-500, is_north_american=False, years=2)
tiny = kfre_person(age=60, is_male=False, eGFR=25, uACR=1e-6, is_north_american=False, years=2)
print(f"uACR=-500 -> {neg:.6f};  uACR=1e-6 -> {tiny:.6f};  identical: {np.isclose(neg,tiny)}")
# Expected: identical -> negative silently clamped, valid-looking risk returned

### Public-API summary

| Bug | Public function | User-facing? | Notes |
|-----|-----------------|--------------|-------|
| 1 | `kfre_person`, `add_kfre_risk_col` | **Yes** | crashes with `UnboundLocalError` |
| 2 | `predict_kfre` | **Yes** | returns `None` instead of raising |
| 3 | `upcr_uacr` only | **Partial** | `kfre_person` already rejects bad sex; only batch coerces |
| 4 | `kfre_person` | **Yes** | negative uACR silently clamped |

Response-letter stance: concede 1, 2, 4 as real user-facing defects (now fixed); for 3, note `kfre_person` already rejects non-boolean sex, so the coercion was specific to the batch `upcr_uacr` path, which has been made explicit.